# 5-2 - Topic modeling avec BERTopic (analyse thématique)

In [1]:
# TODO : se passer de dataset ?

In [2]:
from datasets import load_from_disk
import pandas as pd
import numpy as np
from umap import UMAP  # attention umap-learn pour l'installation
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from stopwordsiso import stopwords
from pathlib import Path
# from bertopic.vectorizers import ClassTfidfTransformer


DATASET_DIR = Path(
    "../models/embeddings/qwen3-8b_embeddings_2026-07-21/dataset_with_embeddings"
)

RANDOM_SEED = 42

# ----- Charger docs et precomputed embeddings -----

# dataset = load_from_disk(DATASET_DIR)
# docs = dataset[
#     "texte"
# ]  # NOTE: list pas obligatoire mais assure ? list(dataset["texte"])
# precomputed_embeddings = np.array(
#     dataset["embedding"]
# )  # Shape : XXX * 4096 pour qwen3-embedding:8b

# ----- Charger docs et precomputed embeddings + pandas -----
# TODO: Si tout en pandas pour faciliter les manipulations
dataset = load_from_disk(DATASET_DIR)

df = dataset.to_pandas()

df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"].astype(str),
    format="%Y%m%d%H%M%S%f",
)

df["dateSeance_day"] = df["dateSeance_ts"].dt.normalize()

df["affiliation_et_gouv"] = df["affiliation_et_gouv"].fillna("Inconnu")

docs = df["texte"].tolist()
precomputed_embeddings = np.array(df["embedding"].tolist())  # reconversion nécessaire
timestamps = df["dateSeance_ts"].tolist()
affiliations = df["affiliation_et_gouv"].tolist()


# ----- Paramètres -----

# UMAP
umap_model = UMAP(
    n_neighbors=15,  # default=15
    n_components=5,  # default=5
    min_dist=0.0,  # default=0.0
    metric="cosine",  # default="cosine"
    random_state=RANDOM_SEED,  # pour reproductibilité
)

# HDBSCAN
# TODO : implémenter et tweaker si besoin

# VECTORIZER
french_stopwords = list(stopwords("fr"))
vectorizer_model = CountVectorizer(stop_words=french_stopwords)

# CTFIDF
# Si besoin réduire impact des mots fréquents, utiliser ClassTfidfTransformer
# ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# NOTE : https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html#removing-stop-words#


# ----- Créer le modèle -----

# initialiser le modèle avec paramètres
topic_model = BERTopic(
    language="french",  # affecte l'affichage des caractères
    embedding_model=None,  # pas d'embedding_model puisque pré-calculés
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,  # enlever les stopwords après embbedings/clustering
    # ctfidf_model=ctfidf_model, # TODO : ou sinon utiliser ctfidf = reduce the impact of frequent word
)

# Fit + transform en utilisant les precomputed_embeddings
topics, probs = topic_model.fit_transform(
    documents=docs, embeddings=precomputed_embeddings
)

W1124 01:19:29.578000 42611 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [ ]:
# TODO : fignoler pour fixer le souci d'env.
# IE : pb reproductibilité UMAP entre uv et conda :
# BLAS / LAPACK : Accelerate vs OpenBLAS
# ie l'aléatoire n'est pas parfaitement reproductible entre les deux environnements, même avec random_state fixé.
# l'initialisation spectrale d'UMAP (décomposition en valeurs propres)
# et les calculs de distances diffèrent dans les résultats de réduction de dimensionnalité, même avec le même random_state.
# Cela peut entraîner des variations dans les topics générés par BERTopic.

# import numpy as np
# from bertopic.dimensionality import BaseDimensionalityReduction

# # UMAP
# umap_model = UMAP(
#     n_neighbors=15,  # default=15
#     n_components=5,  # default=5
#     min_dist=0.0,  # default=0.0
#     metric="cosine",  # default="cosine"
#     random_state=RANDOM_SEED,  # pour reproductibilité
# )

# reduced = umap_model.fit_transform(precomputed_embeddings)
# np.save("../models/embeddings/umap_nn15_nc5_md0.0_cosine_seed42.npy", reduced)


# reduced = np.load(
#     "../models/embeddings/umap_nn15_nc5_md0.0_cosine_seed42.npy"
# )  # sortie de l'ancien env

# topic_model = BERTopic(
#     language="french",  # affecte l'affichage des caractères
#     embedding_model=None,  # pas d'embedding_model puisque pré-calculés
#     umap_model=BaseDimensionalityReduction(),  # n'applique aucune réduction puisque également pré-calculé
#     vectorizer_model=vectorizer_model,  # enlever les stopwords après embbedings/clustering
# )

# # Fit + transform en utilisant les reduced dimension (pré-calculées sur les precomputed_embeddings)
# topics, probs = topic_model.fit_transform(documents=docs, embeddings=reduced)

# Tests

In [4]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_documents(df["Texte_clean"].to_list())

## Topics

In [5]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2381,-1_république_ministre_france_monsieur,"[république, ministre, france, monsieur, franç...","[Depuis trop longtemps, la France n'avait plus..."
1,0,330,0_république_républicain_ve_honte,"[république, républicain, ve, honte, républica...","[Ce ne sera plus une république !, La Républiq..."
2,1,244,1_associations_contrat_association_engagement,"[associations, contrat, association, engagemen...",[Je suis de ceux qui pensent que l'article 6 e...
3,2,213,2_justice_magistrats_judiciaire_tribunaux,"[justice, magistrats, judiciaire, tribunaux, p...","[« À l'heure où nous sommes, les susceptibilit..."
4,3,145,3_mayotte_mahorais_comores_territoire,"[mayotte, mahorais, comores, territoire, situa...",[Il est difficile de passer après M. Lecoq et ...
...,...,...,...,...,...
203,202,11,202_49_censure_gouvernement_constitution,"[49, censure, gouvernement, constitution, moti...",[La démocratie est fragile ; il ne faut pas cr...
204,203,11,203_préambule_alinéa_peuples_libre,"[préambule, alinéa, peuples, libre, mer, déter...",[La réponse qui a été apportée tout à l'heure ...
205,204,10,204_secteurs_charges_entreprises_activité,"[secteurs, charges, entreprises, activité, par...","[Il y a six mois, la France enregistrait ses m..."
206,205,10,205_mer_ultramarins_territoires_martinique,"[mer, ultramarins, territoires, martinique, te...",[Je remercie tous les orateurs – Christophe Eu...


## Barchart

In [6]:
topic_model.visualize_barchart(
    # n_words=10,  # Select the number of words to display per topic
    # topics = [0,1,2,3,4], # Select specific topics to display
    top_n_topics=8,  # Select the first n topics to display
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot
)

In [7]:
## Visualisation documents / topics

In [8]:
topic_model.visualize_documents(
    docs=docs,
    embeddings=precomputed_embeddings,
    hide_annotations=True,  # better readability
    topics=[0, 1, 2, 3],  # Select topics to highlight
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot
)

In [9]:
topic_model.visualize_topics()

## Hiérarchie des topics

In [10]:
topic_model.visualize_hierarchy()

In [11]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
print(topic_model.get_topic_tree(hierarchical_topics))

100%|██████████| 206/206 [00:00<00:00, 470.31it/s]


.
├─école_élèves_scolaire_éducation_enfants
│    ├─école_élèves_éducation_scolaire_aesh
│    │    ├─■──directeurs_école_directeur_directrices_décharge ── Topic: 148
│    │    └─école_élèves_éducation_scolaire_aesh
│    │         ├─école_élèves_aesh_éducation_scolaire
│    │         │    ├─■──parents_école_élèves_vigilants_enseignants ── Topic: 151
│    │         │    └─école_élèves_aesh_scolaire_éducation
│    │         │         ├─école_élèves_éducation_enseignants_scolaire
│    │         │         │    ├─■──école_maternelle_confiance_enseignants_élèves ── Topic: 37
│    │         │         │    └─élèves_école_éducation_scolaire_enseignants
│    │         │         │         ├─■──élèves_école_éducation_scolaire_mathématiques ── Topic: 38
│    │         │         │         └─■──école_éducation_élèves_scolaire_enseignants ── Topic: 36
│    │         │         └─■──aesh_handicap_élèves_enfants_inclusive ── Topic: 11
│    │         └─élèves_classes_école_rentrée_classe
│    │             

In [12]:
topic_model.visualize_heatmap()

## Sauvegarder

In [13]:
# topic_model.save(
#     path="../models/bertopic/qwen3-emb-8B-bertopic-default-with-ctfidf",
#     serialization="safetensors",
#     save_ctfidf=True,
# )

In [14]:
# topic_model = BERTopic.load("../models/bertopic/qwen3-emb-8B-bertopic-default-with-ctfidf")

# Vrac

Comparaison de modèles :

https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html#finding-similar-topics-between-models

## Dynamic topic model

* `global_tuning`
  * Whether to average the topic representation of a topic at time *t* with its global topic representation
* `evolution_tuning`
  * Whether to average the topic representation of a topic at time *t* with the topic representation of that topic at time *t-1*
* `nr_bins`
  * The number of bins to put our timestamps into. It is computationally inefficient to extract the topics at thousands of different timestamps. Therefore, it is advised to keep this value below 20.


In [15]:
# NOTE : timestamps = df["dateSeance_day"].tolist()
# Analyse temporelle
topics_over_time = topic_model.topics_over_time(
    docs=docs,
    timestamps=timestamps,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=20,
)

fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=10,
)

fig_dynamic_topic

In [16]:
# fig_dynamic_topic.write_html("../reports/figures/dynamic_topics.html")

## Topic reduction

In [17]:
# topics_to_merge = [[X, Y],
#                    [Z, W]]
# topic_model.merge_topics(df["Texte_clean"], topics_to_merge)

In [18]:
# DONT : # topic_model.reduce_topics(df["Texte_clean"], nr_topics=40) # DONT, IT NUKES THE TOPICS IN MODEL
# # Access updated topics
# topics = topic_model.topics_

### Topics per class

In [19]:
# ATTENTION, PLANTAIT À CAUSE DES NA
# cf ajout précédent d'une classe "Inconnu" pour les NA dans affiliation_et_gouv
topics_per_class = topic_model.topics_per_class(docs, classes=affiliations)

fig_topics_per_class = topic_model.visualize_topics_per_class(
    topics_per_class,
    # topics=[0, 1, 2, 3],  # choix spécifique de topics à visualiser
    top_n_topics=10,  # choix des X principaux topics
)
fig_topics_per_class

In [20]:
# fig_topics_per_class.write_html("../reports/figures/topics_per_class.html")

In [21]:
# TODO: revoir regroupement des topics
# TODO: revoir Topic distribution
# TODO: aviser alternatives representation sur le nom des topics ?

In [22]:
# df["topic"] = topics
# df["probability"] = probs

# # joindre le nom/label du topic plutôt que juste l'ID
# topic_info = topic_model.get_topic_info()[["Topic", "Name"]]
# df = df.merge(topic_info, left_on="topic", right_on="Topic", how="left")

# # df = df.drop(columns=["embedding"], errors="ignore")
# # df.to_csv("../models/embeddings/export_final.csv", index=False)